In [127]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [128]:
df = pd.read_csv("../data/raw/mental_health.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (2000, 17)


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,,No,Medium


In [129]:
# Modifier les noms des colonnes :
print(df.columns.tolist())

df.columns = df.columns.str.strip()
print(df.columns.tolist())
df.dtypes

['Student_ID', ' Age', ' Gender    ', ' Year_of_Study', ' Sleep_Hours', ' Study_Hours_Per_Week', ' GPA ', ' Social_Media_Hours', ' Screen_Time_Hours', ' Physical_Activity', ' Financial_Stress', ' Family_Support', ' Attendance_Rate', ' Part_Time_Job', ' Diet_Quality', ' Previous_Mental_Health_History', ' Mental_Health_Risk']
['Student_ID', 'Age', 'Gender', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Physical_Activity', 'Financial_Stress', 'Family_Support', 'Attendance_Rate', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History', 'Mental_Health_Risk']


Student_ID                         object
Age                                 int64
Gender                             object
Year_of_Study                       int64
Sleep_Hours                        object
Study_Hours_Per_Week              float64
GPA                                object
Social_Media_Hours                 object
Screen_Time_Hours                 float64
Physical_Activity                  object
Financial_Stress                   object
Family_Support                     object
Attendance_Rate                    object
Part_Time_Job                      object
Diet_Quality                       object
Previous_Mental_Health_History     object
Mental_Health_Risk                 object
dtype: object

- changer les type des colonnes :

In [130]:
df["Sleep_Hours"] = pd.to_numeric(df["Sleep_Hours"], errors="coerce").astype("float64")
df["GPA"] = pd.to_numeric(df["GPA"], errors="coerce").astype("float64")
df["Social_Media_Hours"] = pd.to_numeric(df["Social_Media_Hours"], errors="coerce").astype("float64")
df["Screen_Time_Hours"] = df["Screen_Time_Hours"].astype("float64")
df["Financial_Stress"] = pd.to_numeric(df["Financial_Stress"], errors="coerce").astype("float64")
df["Family_Support"] = pd.to_numeric(df["Family_Support"], errors="coerce").astype("float64")
df["Attendance_Rate"] = pd.to_numeric(df["Attendance_Rate"], errors="coerce").astype("float64")


df["Gender"] = df["Gender"].astype("string")
df["Physical_Activity"] = df["Physical_Activity"].astype("string")
df["Part_Time_Job"] = df["Part_Time_Job"].astype("string")
df["Diet_Quality"] = df["Diet_Quality"].astype("string")
df["Previous_Mental_Health_History"] = df["Previous_Mental_Health_History"].astype("string")
df["Mental_Health_Risk"] = df["Mental_Health_Risk"].astype("string")
df.dtypes

Student_ID                                object
Age                                        int64
Gender                            string[python]
Year_of_Study                              int64
Sleep_Hours                              float64
Study_Hours_Per_Week                     float64
GPA                                      float64
Social_Media_Hours                       float64
Screen_Time_Hours                        float64
Physical_Activity                 string[python]
Financial_Stress                         float64
Family_Support                           float64
Attendance_Rate                          float64
Part_Time_Job                     string[python]
Diet_Quality                      string[python]
Previous_Mental_Health_History    string[python]
Mental_Health_Risk                string[python]
dtype: object

# - Separer les Features et la Target :

In [131]:
# Supprition l'identifiant la colonne "Student_ID" :

df = df.drop(columns=["Student_ID"])

In [132]:
# separer data de X et y :

X = df.drop(columns=["Mental_Health_Risk"])

y = df["Mental_Health_Risk"]

print("-------------------------")
print("la taille de x est :",X.shape)
print("la taille de y est :",y.shape)
print("-------------------------")



-------------------------
la taille de x est : (2000, 15)
la taille de y est : (2000,)
-------------------------


# - Train Test Split :

In [133]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.20, random_state=42, stratify=y)
print("-------------------------")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("-------------------------")
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("-------------------------")


-------------------------
X_train: (1600, 15)
X_test: (400, 15)
-------------------------
y_train: (1600,)
y_test: (400,)
-------------------------


# -Detection des Outliers avec IQR :

In [134]:
# Selectionner les variables numeriques
numeric_columns = X_train.select_dtypes(include=["int64", "float64"]).columns

outliers_ = []
    
for col in numeric_columns:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = X_train[
        (X_train[col] < lower_bound) |
        (X_train[col] > upper_bound)
    ]

    outliers_.append({
        "Column": col,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outliers_Count": len(outliers)
    })

outliers_df = pd.DataFrame(outliers_)

outliers_df

,Column,Lower_Bound,Upper_Bound,Outliers_Count
0,Age,15.50,27.50,2
1,Year_of_Study,-3.50,8.50,0
2,Sleep_Hours,3.75,9.75,9
3,Study_Hours_Per_Week,4.00,36.00,7
4,GPA,1.60,3.52,5
5,Social_Media_Hours,3.05,9.05,10
6,Screen_Time_Hours,3.95,13.15,7
7,Financial_Stress,0.15,11.75,0
8,Family_Support,1.00,12.20,0
9,Attendance_Rate,65.35,104.95,10


# - Difinir les colonnes selon leur type

In [135]:
numeric_features = [
    "Age",
    "Year_of_Study",
    "Sleep_Hours",
    "Study_Hours_Per_Week",
    "GPA",
    "Social_Media_Hours",
    "Screen_Time_Hours",
    "Financial_Stress",
    "Family_Support",
    "Attendance_Rate"
]

nominal_features = [
    "Gender",
    "Part_Time_Job",
    "Previous_Mental_Health_History"
]

ordinal_features = [
    "Physical_Activity",
    "Diet_Quality"
]

# - Créer les Pipelines de Pretraitement

In [138]:

numeric_transfermer = Pipeline([

    ("imputer",SimpleImputer(strategy="median")),

    ("scller",StandardScaler())
])

nominal_transfermer = Pipeline([

    ("imputer",SimpleImputer(strategy="most_frequent")),

    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

ordinal_transfermer = Pipeline([

    ("imputer",SimpleImputer(strategy="most_frequent")),

    ("encoder", OrdinalEncoder(categories=[
            ["Low", "Moderate", "High"],
            ["Poor", "Average", "Good"]
        ],
         handle_unknown="use_encoded_value",
         unknown_value=-1
    ))
])



preprocessor = ColumnTransformer([

    ("numiric",numeric_transfermer,numeric_features),

    ("nominal",nominal_transfermer,nominal_features),

    ('ordinal',ordinal_transfermer,ordinal_features)

])


print(preprocessor)



ColumnTransformer(transformers=[('numiric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scller', StandardScaler())]),
                                 ['Age', 'Year_of_Study', 'Sleep_Hours',
                                  'Study_Hours_Per_Week', 'GPA',
                                  'Social_Media_Hours', 'Screen_Time_Hours',
                                  'Financial_Stress', 'Family_Support',
                                  'Attendance_Rate']),
                                ('nominal',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(s...
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'Part_Time_Job',
                                  'Previous_Mental_Health_H